In [ ]:
import requests
import pandas as pd
import pdfplumber
import re
import io
from bs4 import BeautifulSoup

SOURCES = {
    "KDIGO": {
        "url": "https://kdigo.org/wp-content/uploads/2024/03/KDIGO-2024-CKD-Guideline.pdf",
        "type": "pdf"
    },
    "KDOQI": {
        "url": "https://pubmed.ncbi.nlm.nih.gov/32829751/",
        "type": "html"
    },
    "NIDDK": {
        "url": "https://www.niddk.nih.gov/health-information/kidney-disease/chronic-kidney-disease-ckd/healthy-eating-adults-chronic-kidney-disease",
        "type": "html"
    }
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; CKDNutritionResearch/1.0)"
}

DISEASE = "Kidney Disease (CKD)"

def extract_text(url, file_type):

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=60
    )

    response.raise_for_status()

    if file_type == "pdf":

        text = ""

        with pdfplumber.open(
            io.BytesIO(response.content)
        ) as pdf:

            for page in pdf.pages:
                text += (page.extract_text() or "") + "\n"

        return text

    else:

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup([
            "script",
            "style",
            "nav",
            "footer"
        ]):
            tag.decompose()

        return soup.get_text(
            separator=" ",
            strip=True
        )

PATTERNS = {

    "Protein": [
        r"protein intake.{0,100}?(\d+(?:\.\d+)?)\s*g\s*/\s*kg",
        r"protein.{0,100}?(\d+(?:\.\d+)?)\s*g/kg"
    ],

    "Sodium": [
        r"sodium.{0,100}?(?:<|less than|no more than|up to)\s*(\d[\d,]*)\s*mg",
        r"limit dietary sodium to\s*(\d[\d,]*)\s*mg"
    ],

    "Energy": [
        r"energy intake.{0,100}?(\d+)\s*[-–to]+\s*(\d+)\s*kcal"
    ],

    "Potassium": [],

    "Phosphorus": [],

    "Calcium": [],

    "Total fat": [],

    "Carbohydrates": [],

    "Water": []
}

def extract_nutrients(text, source):

    results = []

    text = re.sub(r"\s+", " ", text)

    for nutrient, patterns in PATTERNS.items():

        found = False

        for pattern in patterns:

            matches = re.finditer(
                pattern,
                text,
                re.IGNORECASE
            )

            for match in matches:

                values = match.groups()

                if nutrient == "Protein":

                    value = float(values[0])

                    minimum = value
                    maximum = value

                    unit = "g/kg body weight/day"

                elif nutrient == "Sodium":

                    value = float(
                        values[0].replace(",", "")
                    )

                    minimum = ""
                    maximum = value

                    unit = "mg/day"

                elif nutrient == "Energy":

                    minimum = float(values[0])
                    maximum = float(values[1])

                    unit = "kcal/day"

                else:
                    continue

                results.append({

                    "disease": DISEASE,

                    "nutrient": nutrient,

                    "min-value": minimum,

                    "max-value": maximum,

                    "unit": unit,

                    "age-group": "Adults",

                    "gender": "All"

                })

                found = True

                break

            if found:
                break

    return results

def add_individualized_nutrients(text):

    results = []

    nutrients = {
        "Potassium": "mg/day",
        "Phosphorus": "mg/day",
        "Calcium": "mg/day",
        "Water": "ml/day"
    }

    text = text.lower()

    for nutrient, unit in nutrients.items():

        if nutrient.lower() in text:

            results.append({

                "disease": DISEASE,

                "nutrient": nutrient,

                "min-value": "",

                "max-value": "",

                "unit": unit,

                "age-group": "Adults",

                "gender": "All"

            })

    return results

all_results = []
source_status = []

for source_name, details in SOURCES.items():

    print(f"Scraping {source_name}...")

    try:

        text = extract_text(
            details["url"],
            details["type"]
        )

        extracted = extract_nutrients(
            text,
            source_name
        )

        individualized = add_individualized_nutrients(
            text
        )

        source_results = extracted + individualized

        all_results.extend(source_results)

        source_status.append({

            "source": source_name,

            "url": details["url"],

            "status": "Success",

            "records_found": len(source_results)

        })

        print(
            f"{source_name}: "
            f"{len(source_results)} records extracted"
        )

    except Exception as error:

        print(
            f"Error scraping {source_name}: {error}"
        )

        source_status.append({

            "source": source_name,

            "url": details["url"],

            "status": str(error),

            "records_found": 0

        })

columns = [
    "disease",
    "nutrient",
    "min-value",
    "max-value",
    "unit",
    "age-group",
    "gender"
]

df = pd.DataFrame(
    all_results,
    columns=columns
)

# Remove duplicate records
df.drop_duplicates(inplace=True)

df.to_csv(
    "CKD_Nutritional_Requirements.csv",
    index=False
)

pd.DataFrame(source_status).to_csv(
    "CKD_Source_Audit.csv",
    index=False
)

print("\nScraping completed!")

print(
    "\nTotal records:",
    len(df)
)

print(
    "\nCSV saved as:",
    "CKD_Nutritional_Requirements.csv"
)

print(df.to_string(index=False))
